## Import thư viện

In [1]:
from pathlib import Path
import os
import json

import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.model_selection import train_test_split

## Khai báo cấu hình dự án

In [2]:
PROJECT_TITLE = "Phân cụm tin tuyển dụng tại Việt nam dựa trên mô tả công việc, yêu cầu ứng viên và đặc điểm tuyển dụng"
DATASET_ID = "tinixai/vietnamese-job-descriptions"

TRAIN_SIZE = 0.9
TEST_SIZE = 0.1
RANDOM_STATE = 42 # Giúp kết quả chia dữ liệu có thể tái lập - mỗi lần chạy sẽ cho cùng một kết quả

print("Project: ", PROJECT_TITLE)
print("Dataset ID: ", DATASET_ID)
print("Train Size: ", TRAIN_SIZE)
print("Test Size: ", TEST_SIZE)

Project:  Phân cụm tin tuyển dụng tại Việt nam dựa trên mô tả công việc, yêu cầu ứng viên và đặc điểm tuyển dụng
Dataset ID:  tinixai/vietnamese-job-descriptions
Train Size:  0.9
Test Size:  0.1


## Khai báo đường dẫn thư mục

In [3]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
METRIC_DIR = OUTPUT_DIR / "metrics"

MODEL_DIR = PROJECT_ROOT / "models"

folders = [
    DATA_DIR,
    RAW_DIR,
    CLEAN_DIR,
    PROCESSED_DIR,
    OUTPUT_DIR,
    FIGURE_DIR,
    TABLE_DIR,
    METRIC_DIR,
    MODEL_DIR,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    
print("Project root: ", PROJECT_ROOT)
print("Raw data folder: ", RAW_DIR)

Project root:  D:\DataScientFinalProject
Raw data folder:  D:\DataScientFinalProject\data\raw


## Tải dataset từ Hugging Face

In [4]:
# Download dataset from Hugging Face
dataset = load_dataset(DATASET_ID, split="train")
dataset

README.md: 0.00B [00:00, ?B/s]

d:\DataScientFinalProject\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\luong\.cache\huggingface\hub\datasets--tinixai--vietnamese-job-descriptions. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data.parquet:   0%|          | 0.00/293M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/606878 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'job_title', 'company_name', 'salary', 'location', 'job_type', 'job_industry', 'experience_level', 'education_level', 'job_position', 'job_description', 'benefits', 'requirements', 'year'],
    num_rows: 606878
})

## Kiểm tra danh sách cột

In [5]:
column_names = dataset.column_names

print("Number of columns:", len(column_names))
print("Columns:")
for col in column_names:
    print("-", col)

Number of columns: 14
Columns:
- id
- job_title
- company_name
- salary
- location
- job_type
- job_industry
- experience_level
- education_level
- job_position
- job_description
- benefits
- requirements
- year


## Kiểm tra tổng quan dữ liệu sau khi tải về

In [6]:
df = dataset.to_pandas()

print("Shape: ", df.shape)
df.head()

print("Number of rows: ", df.shape[0])
print("Number of columns: ", df.shape[1])

display(df.head())
display(df.info())

# Kiểm tra các cột bắt buộc 
expected_columns = [
    "job_title",
    "company_name",
    "salary",
    "location",
    "job_type",
    "job_industry",
    "experience_level",
    "education_level",
    "job_position",
    "job_description",
    "benefits",
    "requirements",
    "year",
]

missing_columns = [col for col in expected_columns if col not in df.columns]

if len(missing_columns) == 0:
    print("Tất cả các cột bắt buộc đều có mặt trong dataset.")
else:
    print("Các cột sau bị thiếu trong dataset:", missing_columns)
    
## Kiểm tra dữ liệu trùng lặp
duplicate_count = df.duplicated().sum()

print(f"Số lượng bản ghi trùng lặp: {duplicate_count}") 
print(f"Tỷ lệ bản ghi trùng lặp: {duplicate_count / len(df) * 100:.2f}%")

## Kiểm tra dữ liệu trùng lặp 
duplicate_count = df.duplicated().sum()

print("Number of duplicated rows:", duplicate_count)
print("Duplicated ratio:", duplicate_count / len(df))

# Kiểm tra missing value tổng quan 
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_ratio": df.isna().mean()
}).sort_values("missing_ratio", ascending=False)

display(missing_summary)

Shape:  (606878, 14)
Number of rows:  606878
Number of columns:  14


,id,job_title,company_name,salary,location,job_type,job_industry,experience_level,education_level,job_position,job_description,benefits,requirements,year
0,1,Java Software Engineer - MSCV 233-2,CÔNG TY TNHH QUICK VIỆT NAM,26 - 36 triệu,Hồ Chí Minh,Toàn thời gian,IT phần mềm,3 năm,Cao đẳng,Nhân Viên,Công ty vốn Nhật chuyên cung cấp dịch vụ điện ...,"Lương: 1,000 ~ 1,400 USD (Gross) Phúc lợi: - B...",Yêu cầu: - Từ 25 ~ 39 tuổi - Trên 3 năm kinh n...,2026
1,2,Software Engineer (Backend/Frontend) - Tiếng T...,CÔNG TY TNHH SONIC FUSION,15 - 35 triệu,Hồ Chí Minh,Toàn thời gian,IT phần mềm,Dưới 1 năm,Không,Nhân Viên,Đọc hiểu tài liệu kỹ thuật và làm việc với đối...,Chế độ bảo hiểm và nghỉ phép đầy đủ theo quy đ...,Thành thạo ít nhất một trong các ngôn ngữ: Pyt...,2026
2,3,"Lead Software Engineer – Desktop (Remote, Engl...",CÔNG TY TNHH PHẦN MỀM GIÁ TRỊ TÍNH TOÁN VÀ ỨNG...,4000 usd,Hà Nội,Toàn thời gian,IT phần mềm,5 năm,Không,Trưởng Nhóm,This role is ideal for a senior developer who ...,Flexible working style Collaborate with divers...,Qualifications Very strong computer science fu...,2026
3,4,Lâp Trình Viên/ Software Engineer/ IT,CÔNG TY TNHH ĐẦU TƯ THƯƠNG MẠI VÀ DỊCH VỤ PHÁT...,10 - 15 triệu,Hà Nội,Toàn thời gian,IT phần mềm,Không,Không,Nhân Viên,"Tham gia vào việc review code, đảm bảo chất lư...",Cơ hội được đào tạo và phát triển bản thân tro...,Có hiểu biết về cấu trúc dữ liệu và giải thuật...,2026
4,5,Software Engineer (Fintech),CÔNG TY TNHH CASSO,14 - 20 triệu,Hồ Chí Minh,Toàn thời gian,IT phần mềm,1 năm,Không,Nhân Viên,Chuẩn hóa dữ liệu: Phối hợp cùng đội ngũ Backe...,Cơ hội lên lead nhanh nếu chứng minh được năng...,Kỹ năng lập trình: Thành thạo ít nhất một ngôn...,2026


<class 'pandas.DataFrame'>
RangeIndex: 606878 entries, 0 to 606877
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                606878 non-null  int64
 1   job_title         606858 non-null  str  
 2   company_name      606772 non-null  str  
 3   salary            606877 non-null  str  
 4   location          606869 non-null  str  
 5   job_type          606878 non-null  str  
 6   job_industry      606878 non-null  str  
 7   experience_level  606878 non-null  str  
 8   education_level   606878 non-null  str  
 9   job_position      606431 non-null  str  
 10  job_description   606851 non-null  str  
 11  benefits          602941 non-null  str  
 12  requirements      603142 non-null  str  
 13  year              606878 non-null  int64
dtypes: int64(2), str(12)
memory usage: 1.2 GB


None

Tất cả các cột bắt buộc đều có mặt trong dataset.
Số lượng bản ghi trùng lặp: 0
Tỷ lệ bản ghi trùng lặp: 0.00%
Number of duplicated rows: 0
Duplicated ratio: 0.0


,missing_count,missing_ratio
benefits,3937,0.006487
requirements,3736,0.006156
job_position,447,0.000737
company_name,106,0.000175
job_description,27,0.000044
job_title,20,0.000033
location,9,0.000015
salary,1,0.000002
job_type,0,0.000000
id,0,0.000000


## Chia train/test theo tỷ lệ 90/10

Với clustering, cần phân bổ `job_industry` đều giữa train/test để đảm bảo tính đại diện của dữ liệu trong cả hai tập.

In [7]:
try:
    raw_train, raw_test = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=df["job_industry"]
    )
    split_method = "stratified_by_job_industry"
except Exception as e:
    print("Stratified split failed:", e)
    print("Fallback to random split.")

    raw_train, raw_test = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True
    )
    split_method = "random_split"

print("Split method:", split_method)
print("Raw train shape:", raw_train.shape)
print("Raw test shape:", raw_test.shape)

Stratified split failed: The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: ['AI Software & Services', 'An ninh - Bảo vệ / Bán sỉ - Bán lẻ - Quản lý cửa hàng / Nghề nghiệp khác', 'An ninh - Bảo vệ / Bất động sản / An toàn lao động', 'An ninh - Bảo vệ / Bất động sản / Lao động phổ thông', 'An ninh - Bảo vệ / Chăm sóc khách hàng', 'An ninh - Bảo vệ / Giáo dục - Đào tạo / Lao động phổ thông', 'An ninh - Bảo vệ / IT Phần cứng - Mạng / Điện - Điện tử - Điện lạnh', 'An ninh - Bảo vệ / Khách sạn - Nhà hàng - Du lịch / Bất động sản', 'An ninh - Bảo vệ / Khách sạn - Nhà hàng - Du lịch / Nghề nghiệp khác', 'An ninh - Bảo vệ / Khách sạn - Nhà hàng - Du lịch / Sản xuất - Lắp ráp - Chế biến', 'An ninh - Bảo vệ / Kế toán / Quản lý dự án', 'An ninh - Bảo vệ / Lao động phổ thông / Khách sạn - Nhà hàng - Du lịch', 'An ninh - Bảo vệ / Nghề nghiệp khác / Lao động phổ thông', 'An ninh - 

## Kiểm tra phân bổ train/test sau khi chia

In [8]:
print("Total rows:", len(df))
print("Train rows:", len(raw_train))
print("Test rows:", len(raw_test))

print("Train ratio:", round(len(raw_train) / len(df), 4))
print("Test ratio:", round(len(raw_test) / len(df), 4))

Total rows: 606878
Train rows: 546190
Test rows: 60688
Train ratio: 0.9
Test ratio: 0.1


## So sánh phân bố ngành nghề train/test

**Mục đích**: Chứng minh rằng phân bố ngành nghề (job_industry) trong tập train và test là tương đồng, đảm bảo tính đại diện của dữ liệu trong cả hai tập.

In [9]:
if "job_industry" in df.columns:
    industry_distribution = pd.DataFrame({
        "train_count": raw_train["job_industry"].value_counts(dropna=False),
        "test_count": raw_test["job_industry"].value_counts(dropna=False),
        "train_ratio": raw_train["job_industry"].value_counts(dropna=False, normalize=True),
        "test_ratio": raw_test["job_industry"].value_counts(dropna=False, normalize=True),
    }).fillna(0)
    
    industry_distribution["abs_ratio_diff"] = (
        industry_distribution["train_ratio"] - industry_distribution["test_ratio"]
    ).abs()

    industry_distribution = industry_distribution.sort_values("train_count", ascending=False)

    display(industry_distribution.head(20))

,train_count,test_count,train_ratio,test_ratio,abs_ratio_diff
job_industry,,,,,
Bán hàng - Kinh doanh,52977.0,5955.0,0.096994,0.098125,0.001131
Chăm sóc khách hàng,50531.0,5552.0,0.092515,0.091484,0.001031
Kế toán / Kiểm toán,45218.0,5024.0,0.082788,0.082784,0.000004
Xây dựng,26827.0,2930.0,0.049117,0.048280,0.000837
Chưa xác định,23057.0,2581.0,0.042214,0.042529,0.000315
Cơ khí - Ô tô - Tự động hóa / Sản xuất - Lắp ráp - Chế biến,22371.0,2490.0,0.040958,0.041030,0.000071
Khoa học - Kỹ thuật,21595.0,2406.0,0.039538,0.039645,0.000108
Nghề nghiệp khác,20985.0,2345.0,0.038421,0.038640,0.000220
Lao động phổ thông,20985.0,2300.0,0.038421,0.037899,0.000522


## So sánh phân bố năm train/test
**Mục đích**: Xem train/test có phân bố năm tương đối giống nhau không.

In [10]:
if "year" in df.columns:
    year_distribution = pd.DataFrame({
        "train_count": raw_train["year"].value_counts(dropna=False),
        "test_count": raw_test["year"].value_counts(dropna=False),
        "train_ratio": raw_train["year"].value_counts(dropna=False, normalize=True),
        "test_ratio": raw_test["year"].value_counts(dropna=False, normalize=True),
    }).fillna(0)

    year_distribution = year_distribution.sort_index()

    display(year_distribution)

,train_count,test_count,train_ratio,test_ratio
year,,,,
2022,100053,10980,0.183184,0.180925
2023,116761,12913,0.213774,0.212777
2024,142985,16064,0.261786,0.264698
2025,144378,16008,0.264337,0.263775
2026,42013,4723,0.076920,0.077824


## Lưu raw data ra csv

In [11]:
raw_train_path = RAW_DIR / "raw_data_train.csv"
raw_test_path = RAW_DIR / "raw_data_test.csv"

raw_train.to_csv(raw_train_path, index=False, encoding="utf-8-sig")
raw_test.to_csv(raw_test_path, index=False, encoding="utf-8-sig")

print("Saved raw train:", raw_train_path)
print("Saved raw test:", raw_test_path)

Saved raw train: D:\DataScientFinalProject\data\raw\raw_data_train.csv
Saved raw test: D:\DataScientFinalProject\data\raw\raw_data_test.csv


## Kiểm tra lại dữ liệu đã lưu ra csv

In [12]:
print("raw_data_train.csv exists:", raw_train_path.exists())
print("raw_data_test.csv exists:", raw_test_path.exists())

train_size_mb = raw_train_path.stat().st_size / (1024 * 1024)
test_size_mb = raw_test_path.stat().st_size / (1024 * 1024)

print(f"raw_data_train.csv size: {train_size_mb:.2f} MB")
print(f"raw_data_test.csv size: {test_size_mb:.2f} MB")

check_train = pd.read_csv(raw_train_path, nrows=5)
check_test = pd.read_csv(raw_test_path, nrows=5)

display(check_train)
display(check_test)

raw_data_train.csv exists: True
raw_data_test.csv exists: True
raw_data_train.csv size: 1054.23 MB
raw_data_test.csv size: 116.93 MB


,id,job_title,company_name,salary,location,job_type,job_industry,experience_level,education_level,job_position,job_description,benefits,requirements,year
0,301834,Nhân Viên Kinh Doanh - Thu Nhập Đến 30 Triệu (...,Công Ty TNHH Bê Tông Trang Trí Việt Nam,12.000.000 - 30.000.000 VND,"Tầng 5, Tòa nhà Phú Hưng 298 Ung Văn Khiêm | 5...",Toàn thời gian,Xây dựng,6 năm,Cao đẳng,Nhân viên,"Lên kế hoạch tiếp cận, chăm sóc khách hàng là ...",Thu nhập: Từ 12- 30 triệu ( Lương cứng + hoa h...,"Độ tuổi từ 27-40, tốt nghiệp cao đẳng, đại học...",2024
1,477331,NV Pha Chế. Caffe,Caffe,6.000.000 - 10.000.000 VND,880 tỉnh lộ 43 phường bình chiểu,Remote,Bán hàng - Kinh doanh,2 năm,Không,Nhân viên,"Tìm kiếm khách hàng tiềm năng, mở rộng nguồn k...",Lương cơ bản + hoa hồng + thưởng KPI - Được hư...,"Nam/Nữ Từ 22 – 35 tuổi, có đam mê và nhiệt huy...",2025
2,540064,Kỹ Sư Giám Sát Xây Dựng,Công Ty TNHH XD TM DV Không Gian Đẹp,10.000.000 - 15.000.000 VND,"68 Song Hành, Quốc lộ 22, Trung Chánh | 72/3C ...",Toàn thời gian,Xây dựng,5 năm,Đại học,Nhân viên,"• Giám sát, Quản lý các tổ đội, từ phần thô đế...",• Mức lương: 10 - 14tr + Phụ cấp công việc • P...,• Tốt nghiệp Đại học chuyên ngành Xây dựng Dân...,2025
3,114270,"Tuyển Sales, Telesales Cho Công Ty Bhnt Pruden...",Công Ty TNHH Một Thành Viên Đại Lý Bảo Hiểm Gl,5.000.000 - 7.000.000 VND,"153 Cách Mạng Tháng Tám, Phường Hoa Lư, Thành ...",Toàn thời gian,Chưa xác định,1 năm,Trung học,Chưa cập nhật,"Gọi điện thoại, kết nối và lên hẹn với khách h...",Được hưởng đầy đủ chế độ BHXH và nghỉ lễ của n...,"cần ứng viên từ 21 tuổi, tốt nghiệp THPT trở lên",2022
4,322289,Kế Toán Tổng Hợp,CÔNG TY TNHH XUẤT NHẬP KHẨU Ô TÔ MIỀN NAM,8.000.000 - 10.000.000 VND,"159 Nguyễn Chí Thanh | 68 QL1A, Phường An Phú ...",Toàn thời gian,Kế toán / Kiểm toán,5 năm,Cao đẳng,Nhân viên,− Thực hiện hiệu quả sổ sách kế toán các nghiệ...,− Lương cơ bản: 8.000.000 đồng – 10.000.000 đồ...,"− Trình độ: Tốt nghiệp Cao đẳng, Đại học chuyê...",2024


,id,job_title,company_name,salary,location,job_type,job_industry,experience_level,education_level,job_position,job_description,benefits,requirements,year
0,317083,Nhân Viên Đối Ngoại,Công Ty TNHH Đầu Tư Và Phát Triển Nguồn Nhân L...,8.000.000 - 11.000.000 VND,"TT20 Trịnh Văn Bô | Số 17A, Tổ dân phố số 5 - ...",Toàn thời gian,Giáo dục - Đào tạo / Biên phiên dịch,2 năm,Đại học,Nhân viên,Phụ trách công việc: trực chát với đối tác Đài...,Lương: cơ bản (theo thoả thuận) + doanh số xuấ...,"Tốt nghiệp đại học, có chứng chỉ tiếng trung H...",2024
1,384327,Giáo Viên Toeic Full / Part-Time,Công Ty TNHH Một Thành Viên Đoàn Phan Gia Lâm,5.000.000 - 15.000.000 VND,"30 Trần Quang Diệu | 31 Trương Văn Đa, Hòa Khá...",Bán thời gian,Giáo dục - Đào tạo / Chăm sóc khách hàng,1 năm,Không,Nhân viên,Thực hiện công việc giảng dạy theo đúng mục ti...,Lương cơ bản từ 5 triệu cho part-time và 8-15 ...,"Có chứng chỉ TOEIC 850+ hoặc IELTS từ 7.5, phá...",2024
2,509652,Nhân Viên Mua Hàng - Xử Lý Đơn Hàng,Công Ty TNHH Nhựa Song Mộc,10.000.000 - 12.000.000 VND,"Đường số 11, Khu công nghiệp Tân Đức, Xã Hựu T...",Toàn thời gian,Kế toán / Kiểm toán,3 năm,Trung cấp,Nhân viên,"Nhân viên mua hàng, nhận và xử lý đơn hàng, xu...",Tổng thu nhập từ 10 triệu đến 12 triệu đồng ( ...,Độ tuổi từ 22-35t - Trình độ: Tốt nghiệp trung...,2025
3,278737,Nhân Viên Quản Lý Dự Án Chuẩn Bị Sản Xuất Sản ...,Công Ty TNHH Yamaha Motor Việt Nam,10.000.000 - 13.000.000 VND,Nhà máy Yamaha Motor Việt Nam – Khu CN Nội Bài...,Toàn thời gian,Cơ khí - Ô tô - Tự động hóa / Sản xuất - Lắp r...,3 năm,Đại học,Nhân viên,Đề xuất mục tiêu và chuẩn bị kế hoạch thực hiệ...,Địa điểm làm việc: Nhà máy Yamaha Motor Việt N...,"Tốt nghiệp Đại học; - Tiếng Anh tốt, thành thạ...",2023
4,468353,Kỹ Thuật Viên (Không Yêu Cầu Kinh Nghiệm),VPĐD AMAZON PAPYRUS CHEMICALS (VIETNAM) LIMITE...,8.000.000 - 10.000.000 VND,"Cụm CN Phú Lâm/ Phong Khê | Tầng 7, Tòa nhà To...",Toàn thời gian,Chưa xác định,1 năm,Cao đẳng,Nhân viên,"Dịch vụ kỹ thuật, theo dõi hoá chất sử dụng tạ...",Mức lương thỏa thuận theo năng lực - Các chế đ...,"Tốt nghiệp Cao Ðẳng, Ðại học chuyên ngành Công...",2025


## Lưu metadata về dataset đã tải về

In [13]:
metadata = {
    "project_title": PROJECT_TITLE,
    "dataset_id": DATASET_ID,
    "split_method": split_method if "split_method" in globals() else "random_split",
    "random_state": RANDOM_STATE,
    "train_size": TRAIN_SIZE,
    "test_size": TEST_SIZE,
    "n_total_rows": len(df),
    "n_raw_train": len(raw_train),
    "n_raw_test": len(raw_test),
    "n_columns": df.shape[1],
    "columns": ", ".join(df.columns),
}

metadata_df = pd.DataFrame([metadata])

metadata_path = TABLE_DIR / "stage_02_metadata.csv"
metadata_df.to_csv(metadata_path, index=False, encoding="utf-8-sig")

display(metadata_df)
print("Saved metadata:", metadata_path)

,project_title,dataset_id,split_method,random_state,train_size,test_size,n_total_rows,n_raw_train,n_raw_test,n_columns,columns
0,Phân cụm tin tuyển dụng tại Việt nam dựa trên ...,tinixai/vietnamese-job-descriptions,random_split,42,0.9,0.1,606878,546190,60688,14,"id, job_title, company_name, salary, location,..."


Saved metadata: D:\DataScientFinalProject\outputs\tables\stage_02_metadata.csv


## Lưu bảng missing summary và phân bố ngành nghề

In [14]:
missing_summary.to_csv(
    TABLE_DIR / "stage_02_missing_summary_raw.csv",
    encoding="utf-8-sig"
)

if "industry_distribution" in globals():
    industry_distribution.to_csv(
        TABLE_DIR / "stage_02_train_test_industry_distribution.csv",
        encoding="utf-8-sig"
    )

if "year_distribution" in globals():
    year_distribution.to_csv(
        TABLE_DIR / "stage_02_train_test_year_distribution.csv",
        encoding="utf-8-sig"
    )

print("Saved stage 02 summary tables.")

Saved stage 02 summary tables.
